# Reconcile source accounts, one step at a time

This notebook walks through the preparation workflow that motivated CCRT: import
UCDP and ACLED accounts, inspect possible links, and preserve disagreements for
later analysis. All incidents here are **synthetic**, so we can test behavior without
private data. Install `.[tables,dev]` from the repository first.

Run this from the repository root or the `examples` folder. Nothing here uses an API key.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from ccrt import MatchConfig, Review, load_acled_csv, load_ucdp_csv, reconcile
from ccrt.export import read_parquet, write_csv, write_parquet
from ccrt.storage import load_dataset, save_dataset, save_review, save_run

root = Path.cwd()
if root.name == "examples":
    root = root.parent
examples = root / "examples"
assert (examples / "ucdp_synthetic.csv").exists(), "Run from the repo or examples directory"
output = TemporaryDirectory(prefix="ccrt-tutorial-")
work = Path(output.name)
print("Temporary outputs:", work)

## 1. A record is one source's account

A shared schema gives equivalent fields consistent names. It must also preserve
fields that have no equivalent. UCDP records a civilian count; the ACLED export's
total fatalities cannot fill that field. Inspect both the normalized view and raw
account below.

In [ ]:
ucdp = load_ucdp_csv(examples / "ucdp_synthetic.csv", version="26.1")
acled = load_acled_csv(examples / "acled_synthetic.csv", version="synthetic-2026-09-24")
print(ucdp.records[0].raw)
ucdp.to_pandas()[["source_id", "date_start", "date_end", "total_deaths_best", "civilian_deaths"]]

In [ ]:
acled.to_pandas()[["source_id", "total_deaths_best", "civilian_deaths", "date_basis"]]

## 2. Missing information stays visible

A record without coordinates cannot participate in this distance-based search.
It still belongs in the source dataset. Unknown fatality counts also stay distinct
from zero.

In [ ]:
[(issue.row, issue.severity, issue.message) for issue in ucdp.issues]

## 3. Candidate generation is a search

We ask which accounts could describe the same incident, using date intervals and
reported coordinates. Several candidates can refer to one account. Actor names,
original narratives, source coverage, and aggregation rules need human review.

In [ ]:
result = reconcile(ucdp, acled, config=MatchConfig(max_days=3, max_km=25))
for c in result.candidates:
    print(c.left.source_id, c.right.source_id, round(c.score, 3), c.warnings)
print("Unlinked:", [(u.record.source_id, u.reason) for u in result.unlinked])
assert len(result.candidates) == 3

## 4. Record an explicit decision

Our synthetic fixture specifies U1/A1 as the same incident and U1/A2 as distinct.
The matching rules propose both. We can accept one and reject the other while
leaving other candidates unresolved. The original claims never change.

In [ ]:
review = Review.create(result)
for c in result.candidates:
    if (c.left.source_id, c.right.source_id) == ("U1", "A1"):
        review = review.decide(
            c.candidate_id,
            "accepted",
            reviewer="Tutorial",
            note="Synthetic ground truth specifies the same incident",
        )
    elif (c.left.source_id, c.right.source_id) == ("U1", "A2"):
        review = review.decide(
            c.candidate_id,
            "rejected",
            reviewer="Tutorial",
            note="Synthetic ground truth specifies distinct incidents",
        )
[(d.status, d.note) for d in review.decisions]

## 5. Reproduce the result

A run bundle contains both input datasets, fingerprints, and settings. A review is
bound to that run. Parquet includes provenance in its metadata; a CSV is convenient
for inspection. Do not sum casualty columns in a candidate table: accounts repeat.

In [ ]:
save_dataset(ucdp, work / "ucdp.json")
save_run(result, work / "run.json")
save_review(review, result, work / "review.json")
write_csv(result, work / "candidates.csv", review=review)
write_parquet(ucdp, work / "ucdp.parquet")
assert load_dataset(work / "ucdp.json") == ucdp
assert read_parquet(work / "ucdp.parquet") == ucdp
print((work / "candidates.csv").read_text().splitlines()[0])

## 6. Measure the tradeoff

For these invented accounts we know every true pair. Precision asks what fraction
of proposed links are true; recall asks what fraction of true links were proposed.
They measure this synthetic search problem only. Changing the window or allowing
coarse locations changes which pairs become eligible.

In [ ]:
from dataclasses import asdict

from ccrt.evaluation import evaluate_candidates

truth = {("U1", "A1"), ("U2", "A3"), ("U3", "A4")}
for config in [MatchConfig(max_days=0, max_km=1), MatchConfig(), MatchConfig(include_coarse_locations=True)]:
    proposed = reconcile(ucdp, acled, config=config)
    print(config.to_dict(), asdict(evaluate_candidates(proposed, truth)))

## Exercises for our next session

1. Before running it, predict what happens to U1/A1 when `max_days=0`. Explain
   why U2/A3 can still match under that setting.
2. Inspect why A2 receives a strong score despite being a different synthetic
   incident. Which evidence would you want to see before accepting a link?
3. Write a test showing that a new input snapshot invalidates an old review.

For actual research, obtain exports through your permitted source access, choose a
bounded comparison population, and manually label true links before interpreting
metrics. A UCDP token can be supplied via `UCDP_API_KEY`; the API example in the
README is intentionally separate from this offline notebook.

In [ ]:
output.cleanup()